In [41]:
import pandas as pd



In [42]:
dfi = pd.read_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\raw\icustays.csv")
dfi.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10018328,23786647,31269608,Neuro Stepdown,Neuro Stepdown,2154-04-24 23:03:44,2154-05-02 15:55:21,7.702512
1,10020187,24104168,37509585,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Stepdown,2169-01-15 04:56:00,2169-01-20 15:47:50,5.452662
2,10020187,26842957,32554129,Neuro Intermediate,Neuro Intermediate,2170-02-24 18:18:46,2170-02-25 15:15:26,0.872685
3,10012853,27882036,31338022,Trauma SICU (TSICU),Trauma SICU (TSICU),2176-11-26 02:34:49,2176-11-29 20:58:54,3.766725
4,10020740,25826145,32145159,Trauma SICU (TSICU),Trauma SICU (TSICU),2150-06-03 20:12:32,2150-06-04 21:05:58,1.037106


In [43]:
dfi["intime"] = pd.to_datetime(dfi["intime"])

In [44]:
icustays_sorted = dfi.sort_values(
    by=["subject_id", "intime"]
)

first_icustay = icustays_sorted.drop_duplicates(
    subset="subject_id",
    keep="first"
)


In [45]:
first_icustay

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
44,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
66,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
130,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588
53,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2156-04-12 16:24:18,2156-04-17 15:57:08,4.981134
22,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512
...,...,...,...,...,...,...,...,...
27,10038999,27189241,39711498,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2131-05-22 21:50:33,2131-05-31 17:55:04,8.836470
119,10039708,28258130,33281088,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2140-01-23 18:08:00,2140-02-08 22:28:20,16.180787
76,10039831,26924951,39142259,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2115-12-28 19:36:43,2115-12-30 16:31:48,1.871586
78,10039997,24294180,36893762,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2134-09-07 18:03:58,2134-09-08 22:11:05,1.171609


Sanity Check

In [46]:
first_icustay["subject_id"].is_unique


True

In [47]:
dfp = pd.read_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\raw\patients.csv")
dfp.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10014729,F,21,2125,2011 - 2013,NaN
1,10003400,F,72,2134,2011 - 2013,2137-09-02
2,10002428,F,80,2155,2011 - 2013,NaN
3,10032725,F,38,2143,2011 - 2013,2143-03-30
4,10027445,F,48,2142,2011 - 2013,2146-02-09


In [48]:
patients = dfp.drop(["anchor_year", "anchor_year_group", "dod"], axis=1)

In [49]:
patients

,subject_id,gender,anchor_age
0,10014729,F,21
1,10003400,F,72
2,10002428,F,80
3,10032725,F,38
4,10027445,F,48
...,...,...,...
95,10004733,M,51
96,10021118,M,62
97,10018501,M,83
98,10007058,M,48


In [50]:
cohort = first_icustay.merge(
    patients[["subject_id", "gender", "anchor_age"]],
    on="subject_id",
    how="left"
)


In [51]:
cohort = cohort[cohort["anchor_age"] >= 18]


In [52]:
cohort.shape


(100, 10)

In [53]:
cohort["subject_id"].is_unique
cohort["stay_id"].is_unique


True

In [54]:
cohort["anchor_age"].describe()


count    100.00000
mean      61.75000
std       16.16979
min       21.00000
25%       51.75000
50%       63.00000
75%       72.00000
max       91.00000
Name: anchor_age, dtype: float64

In [55]:
cohort.to_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\interim/cohort.csv", index=False)  


In [56]:
dfa = pd.read_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\raw\admissions.csv")
dfa

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,NaN,URGENT,P41R5N,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,?,MARRIED,HISPANIC/LATINO - PUERTO RICAN,NaN,NaN,0
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,NaN,URGENT,P233F6,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,2134-08-17 16:24:00,2134-08-18 03:15:00,0
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,2111-11-15 17:20:00,URGENT,P38TI6,TRANSFER FROM HOSPITAL,DIED,Medicaid,ENGLISH,NaN,UNKNOWN,NaN,NaN,1
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,NaN,URGENT,P07HDB,TRANSFER FROM HOSPITAL,HOME,Other,ENGLISH,WIDOWED,WHITE,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,10038992,24745425,2187-07-29 01:05:00,2187-08-03 17:02:00,NaN,SURGICAL SAME DAY ADMISSION,P41R5N,PHYSICIAN REFERRAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,NaN,NaN,0
271,10008287,22168393,2145-09-28 01:17:00,2145-10-02 13:35:00,NaN,SURGICAL SAME DAY ADMISSION,P898NM,PHYSICIAN REFERRAL,HOME HEALTH CARE,Other,ENGLISH,SINGLE,WHITE,NaN,NaN,0
272,10022880,27708593,2177-03-12 07:15:00,2177-03-19 14:25:00,NaN,SURGICAL SAME DAY ADMISSION,P99698,PHYSICIAN REFERRAL,HOME,Medicare,ENGLISH,MARRIED,WHITE,NaN,NaN,0
273,10004457,23251352,2141-12-17 11:00:00,2141-12-21 15:56:00,NaN,SURGICAL SAME DAY ADMISSION,P41R5N,PHYSICIAN REFERRAL,REHAB,Medicare,ENGLISH,SINGLE,OTHER,NaN,NaN,0


Create the mortality label

In [57]:
dfa["mortality"] = dfa["deathtime"].notna().astype(int)


In [58]:
admissions_reduced = dfa[["hadm_id", "mortality"]]


In [59]:
cohort = cohort.merge(
    admissions_reduced,
    on="hadm_id",
    how="left"
)


In [60]:
cohort["mortality"].isna().sum()


np.int64(0)

In [61]:
cohort["mortality"].value_counts(normalize=True)


mortality
0    0.89
1    0.11
Name: proportion, dtype: float64

In [62]:
cohort["stay_id"].is_unique


True

In [63]:
cohort.to_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\interim/cohort_with_mortality.csv", index=False)


In [64]:
cohort

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,mortality
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,0
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,0
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,F,46,0
3,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2156-04-12 16:24:18,2156-04-17 15:57:08,4.981134,F,80,0
4,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,0
...,...,...,...,...,...,...,...,...,...,...,...
95,10038999,27189241,39711498,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2131-05-22 21:50:33,2131-05-31 17:55:04,8.836470,M,45,0
96,10039708,28258130,33281088,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2140-01-23 18:08:00,2140-02-08 22:28:20,16.180787,F,46,0
97,10039831,26924951,39142259,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2115-12-28 19:36:43,2115-12-30 16:31:48,1.871586,F,57,0
98,10039997,24294180,36893762,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2134-09-07 18:03:58,2134-09-08 22:11:05,1.171609,F,67,0


In [65]:
cohort["intime"] = pd.to_datetime(cohort["intime"])

In [66]:
cohort["obs_endtime"] = cohort["intime"] + pd.Timedelta(hours=24)


In [67]:
cohort

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,mortality,obs_endtime
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,0,2180-07-24 14:00:00
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,0,2157-11-21 19:18:02
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,F,46,0,2110-04-12 15:52:22
3,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2156-04-12 16:24:18,2156-04-17 15:57:08,4.981134,F,80,0,2156-04-13 16:24:18
4,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,0,2141-05-23 20:18:01
...,...,...,...,...,...,...,...,...,...,...,...,...
95,10038999,27189241,39711498,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2131-05-22 21:50:33,2131-05-31 17:55:04,8.836470,M,45,0,2131-05-23 21:50:33
96,10039708,28258130,33281088,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2140-01-23 18:08:00,2140-02-08 22:28:20,16.180787,F,46,0,2140-01-24 18:08:00
97,10039831,26924951,39142259,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2115-12-28 19:36:43,2115-12-30 16:31:48,1.871586,F,57,0,2115-12-29 19:36:43
98,10039997,24294180,36893762,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2134-09-07 18:03:58,2134-09-08 22:11:05,1.171609,F,67,0,2134-09-08 18:03:58


In [68]:
chartevents = pd.read_csv(
    r"C:\Users\user\Desktop\icu_mortality_project\data\raw\chartevents.csv",
    usecols=["stay_id", "itemid", "charttime", "valuenum"]
)

chartevents["charttime"] = pd.to_datetime(chartevents["charttime"])


In [69]:
chartevents = chartevents.merge(
    cohort[["stay_id", "intime", "obs_endtime"]],
    on="stay_id",
    how="inner"
)


In [70]:
chartevents

,stay_id,charttime,itemid,valuenum,intime,obs_endtime
0,32604416,2132-12-16 00:00:00,225054,NaN,2132-12-15 09:29:01,2132-12-16 09:29:01
1,32604416,2132-12-16 00:00:00,223769,100.0,2132-12-15 09:29:01,2132-12-16 09:29:01
2,32604416,2132-12-16 00:00:00,223956,NaN,2132-12-15 09:29:01,2132-12-16 09:29:01
3,32604416,2132-12-16 00:00:00,224866,NaN,2132-12-15 09:29:01,2132-12-16 09:29:01
4,32604416,2132-12-16 00:00:00,227341,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01
...,...,...,...,...,...,...
501467,34107647,2153-03-28 10:49:28,220001,NaN,2153-03-28 02:21:00,2153-03-29 02:21:00
501468,34107647,2153-03-28 10:49:28,220001,NaN,2153-03-28 02:21:00,2153-03-29 02:21:00
501469,34107647,2153-03-28 10:49:28,220001,NaN,2153-03-28 02:21:00,2153-03-29 02:21:00
501470,34107647,2153-03-28 10:49:28,220001,NaN,2153-03-28 02:21:00,2153-03-29 02:21:00


In [72]:
chartevents_24h = chartevents[
    (chartevents["charttime"] >= chartevents["intime"]) &
    (chartevents["charttime"] <= chartevents["obs_endtime"])
]


Load labs

In [73]:
labevents = pd.read_csv(
    r"C:\Users\user\Desktop\icu_mortality_project\data\raw\labevents.csv",
    usecols=["hadm_id", "itemid", "charttime", "valuenum"]
)

labevents["charttime"] = pd.to_datetime(labevents["charttime"])


Attach ICU times (TEMP merge)

In [74]:
labevents = labevents.merge(
    cohort[["hadm_id", "intime", "obs_endtime"]],
    on="hadm_id",
    how="inner"
)


In [75]:
labevents

,hadm_id,itemid,charttime,valuenum,intime,obs_endtime
0,28477280.0,50970,2137-10-16 02:00:00,2.8,2137-10-12 22:44:57,2137-10-13 22:44:57
1,28477280.0,50931,2137-10-16 02:00:00,91.0,2137-10-12 22:44:57,2137-10-13 22:44:57
2,28477280.0,50868,2137-10-16 02:00:00,14.0,2137-10-12 22:44:57,2137-10-13 22:44:57
3,28477280.0,50960,2137-10-16 02:00:00,2.0,2137-10-12 22:44:57,2137-10-13 22:44:57
4,28477280.0,50862,2137-10-16 02:00:00,2.3,2137-10-12 22:44:57,2137-10-13 22:44:57
...,...,...,...,...,...,...
42326,28998349.0,50804,2116-12-07 18:59:00,35.0,2116-12-03 01:02:00,2116-12-04 01:02:00
42327,28998349.0,50818,2116-12-07 18:59:00,56.0,2116-12-03 01:02:00,2116-12-04 01:02:00
42328,28998349.0,52033,2116-12-07 18:59:00,NaN,2116-12-03 01:02:00,2116-12-04 01:02:00
42329,28998349.0,50825,2116-12-07 18:59:00,39.7,2116-12-03 01:02:00,2116-12-04 01:02:00


In [76]:
labevents_24h = labevents[
    (labevents["charttime"] >= labevents["intime"]) &
    (labevents["charttime"] <= labevents["obs_endtime"])
]


Check 1 — No future timestamps

In [77]:
(chartevents_24h["charttime"] > chartevents_24h["obs_endtime"]).sum()
(labevents_24h["charttime"] > labevents_24h["obs_endtime"]).sum()


np.int64(0)

In [78]:
chartevents_24h["stay_id"].isin(cohort["stay_id"]).all()
labevents_24h["hadm_id"].isin(cohort["hadm_id"]).all()


np.True_

In [79]:
chartevents_24h.to_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\interim/chartevents_24h.csv", index=False)
labevents_24h.to_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\interim/labevents_24h.csv", index=False)


In [80]:
d_items = pd.read_csv(
                    r"C:\Users\user\Desktop\icu_mortality_project\data\raw\d_items.csv"
                    ,usecols=["category", "itemid", "label"]
                    )

In [81]:
d_items

,itemid,label,category
0,226228,Gender,ADT
1,226545,Race,ADT
2,229877,Suction events (CH),ECMO
3,229875,Oxygenator visible (CH),ECMO
4,229266,Cannula sites visually inspected (ECMO),ECMO
...,...,...,...
4009,227452,Tobramycin (Trough),Labs
4010,227451,Tobramycin (Random),Labs
4011,227457,Platelet Count,Labs
4012,227468,Fibrinogen,Labs


In [82]:
VITAL_ITEMIDS = {
    "heart_rate": [220045],
    "sbp": [220179],        # Non-invasive systolic BP
    "dbp": [220180],        # Non-invasive diastolic BP
    "resp_rate": [220210],
    "spo2": [220277],
    "temperature": [223761]  # Temperature (°C)
}


In [83]:
selected_itemids = [itemid for ids in VITAL_ITEMIDS.values() for itemid in ids]


In [84]:
selected_itemids

[220045, 220179, 220180, 220210, 220277, 223761]

In [85]:
chartevents_24h

,stay_id,charttime,itemid,valuenum,intime,obs_endtime
0,32604416,2132-12-16 00:00:00,225054,NaN,2132-12-15 09:29:01,2132-12-16 09:29:01
1,32604416,2132-12-16 00:00:00,223769,100.0,2132-12-15 09:29:01,2132-12-16 09:29:01
2,32604416,2132-12-16 00:00:00,223956,NaN,2132-12-15 09:29:01,2132-12-16 09:29:01
3,32604416,2132-12-16 00:00:00,224866,NaN,2132-12-15 09:29:01,2132-12-16 09:29:01
4,32604416,2132-12-16 00:00:00,227341,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01
...,...,...,...,...,...,...
501467,34107647,2153-03-28 10:49:28,220001,NaN,2153-03-28 02:21:00,2153-03-29 02:21:00
501468,34107647,2153-03-28 10:49:28,220001,NaN,2153-03-28 02:21:00,2153-03-29 02:21:00
501469,34107647,2153-03-28 10:49:28,220001,NaN,2153-03-28 02:21:00,2153-03-29 02:21:00
501470,34107647,2153-03-28 10:49:28,220001,NaN,2153-03-28 02:21:00,2153-03-29 02:21:00


In [86]:
vitals_ts = chartevents_24h[
    chartevents_24h["itemid"].isin(selected_itemids)
].copy()


In [87]:
itemid_to_name = {
    220045: "heart_rate",
    220179: "sbp",
    220180: "dbp",
    220210: "resp_rate",
    220277: "spo2",
    223761: "temperature"
}

vitals_ts["vital_name"] = vitals_ts["itemid"].map(itemid_to_name)


In [88]:
vitals_ts

,stay_id,charttime,itemid,valuenum,intime,obs_endtime,vital_name
37,32604416,2132-12-16 00:00:00,220210,19.0,2132-12-15 09:29:01,2132-12-16 09:29:01,resp_rate
69,32604416,2132-12-16 00:00:00,220045,80.0,2132-12-15 09:29:01,2132-12-16 09:29:01,heart_rate
92,32604416,2132-12-16 00:00:00,220277,95.0,2132-12-15 09:29:01,2132-12-16 09:29:01,spo2
102,32604416,2132-12-16 01:00:00,220045,82.0,2132-12-15 09:29:01,2132-12-16 09:29:01,heart_rate
104,32604416,2132-12-16 01:00:00,220277,96.0,2132-12-15 09:29:01,2132-12-16 09:29:01,spo2
...,...,...,...,...,...,...,...
501351,34107647,2153-03-28 02:57:00,220045,99.0,2153-03-28 02:21:00,2153-03-29 02:21:00,heart_rate
501352,34107647,2153-03-28 02:57:00,223761,97.8,2153-03-28 02:21:00,2153-03-29 02:21:00,temperature
501353,34107647,2153-03-28 02:57:00,220277,92.0,2153-03-28 02:21:00,2153-03-29 02:21:00,spo2
501354,34107647,2153-03-28 02:57:00,220180,62.0,2153-03-28 02:21:00,2153-03-29 02:21:00,dbp


In [89]:
vitals_ts["vital_name"].value_counts()


vital_name
resp_rate      2748
heart_rate     2733
spo2           2658
dbp            1554
sbp            1552
temperature     588
Name: count, dtype: int64

In [90]:
vitals_ts["stay_id"].isin(cohort["stay_id"]).all()

np.True_

In [91]:
vitals_ts["valuenum"].isna().mean()


np.float64(0.0)

In [92]:
vitals_ts.to_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\interim/vitals_timeseries_24h.csv", index=False)


In [93]:
vitals_ts

,stay_id,charttime,itemid,valuenum,intime,obs_endtime,vital_name
37,32604416,2132-12-16 00:00:00,220210,19.0,2132-12-15 09:29:01,2132-12-16 09:29:01,resp_rate
69,32604416,2132-12-16 00:00:00,220045,80.0,2132-12-15 09:29:01,2132-12-16 09:29:01,heart_rate
92,32604416,2132-12-16 00:00:00,220277,95.0,2132-12-15 09:29:01,2132-12-16 09:29:01,spo2
102,32604416,2132-12-16 01:00:00,220045,82.0,2132-12-15 09:29:01,2132-12-16 09:29:01,heart_rate
104,32604416,2132-12-16 01:00:00,220277,96.0,2132-12-15 09:29:01,2132-12-16 09:29:01,spo2
...,...,...,...,...,...,...,...
501351,34107647,2153-03-28 02:57:00,220045,99.0,2153-03-28 02:21:00,2153-03-29 02:21:00,heart_rate
501352,34107647,2153-03-28 02:57:00,223761,97.8,2153-03-28 02:21:00,2153-03-29 02:21:00,temperature
501353,34107647,2153-03-28 02:57:00,220277,92.0,2153-03-28 02:21:00,2153-03-29 02:21:00,spo2
501354,34107647,2153-03-28 02:57:00,220180,62.0,2153-03-28 02:21:00,2153-03-29 02:21:00,dbp


In [94]:
vitals_ts['value'] = pd.to_numeric(vitals_ts['valuenum'], errors='coerce')


In [95]:
vitals_ts = vitals_ts.sort_values(['stay_id', 'vital_name', 'charttime'])


In [96]:
vitals_agg = (
    vitals_ts
    .groupby(['stay_id', 'vital_name'])
    .agg(
        mean_value=('value', 'mean'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        last_value=('value', 'last')
    )
    .reset_index()
)


In [97]:
vitals_wide = vitals_agg.pivot(
    index='stay_id',
    columns='vital_name'
)


In [98]:
vitals_wide.columns = [
    f"{vital}_{stat}"
    for stat, vital in vitals_wide.columns
]

vitals_wide = vitals_wide.reset_index()


In [99]:
vitals_wide

,stay_id,dbp_mean_value,heart_rate_mean_value,resp_rate_mean_value,sbp_mean_value,spo2_mean_value,temperature_mean_value,dbp_min_value,heart_rate_min_value,resp_rate_min_value,...,resp_rate_max_value,sbp_max_value,spo2_max_value,temperature_max_value,dbp_last_value,heart_rate_last_value,resp_rate_last_value,sbp_last_value,spo2_last_value,temperature_last_value
0,30057454,54.666667,108.533333,18.266667,79.333333,92.366667,98.057143,47.0,100.0,13.0,...,25.0,91.0,96.0,98.7,55.0,105.0,13.0,79.0,92.0,98.1
1,30101877,69.750000,94.875000,20.541667,137.916667,100.000000,100.071429,49.0,79.0,16.0,...,25.0,166.0,100.0,100.5,76.0,79.0,19.0,147.0,100.0,99.3
2,30458995,65.333333,81.720000,17.600000,129.833333,97.360000,97.966667,45.0,60.0,14.0,...,24.0,157.0,99.0,98.3,60.0,91.0,16.0,143.0,97.0,98.3
3,30585761,59.000000,70.958333,19.375000,139.000000,95.181818,97.950000,59.0,60.0,10.0,...,24.0,139.0,98.0,98.2,59.0,70.0,21.0,139.0,94.0,97.9
4,30665396,NaN,91.346154,20.280000,NaN,98.650000,98.283333,NaN,70.0,10.0,...,36.0,NaN,100.0,98.9,NaN,82.0,22.0,NaN,100.0,98.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,39553978,54.100000,96.500000,20.700000,88.900000,96.300000,98.966667,41.0,91.0,16.0,...,24.0,95.0,99.0,99.5,55.0,94.0,20.0,85.0,95.0,99.5
96,39623478,61.900000,97.000000,15.000000,110.050000,96.550000,98.333333,47.0,84.0,9.0,...,20.0,140.0,99.0,98.7,63.0,95.0,15.0,130.0,93.0,98.7
97,39635619,78.272727,95.352941,21.705882,147.878788,95.735294,100.071429,50.0,82.0,13.0,...,31.0,181.0,100.0,101.3,63.0,82.0,13.0,112.0,94.0,100.8
98,39711498,67.727273,110.960000,20.760000,107.409091,97.000000,99.650000,55.0,29.0,17.0,...,22.0,131.0,100.0,101.7,79.0,121.0,20.0,101.0,100.0,101.7


In [100]:
vitals_wide.to_csv(
    r"C:\Users\user\Desktop\icu_mortality_project\data\interim/vitals_agg.csv",
    index=False
)


In [101]:
d_labitems = pd.read_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\raw\d_labitems.csv")

In [102]:
d_labitems

,itemid,label,fluid,category
0,50808,Free Calcium,Blood,Blood Gas
1,50826,Tidal Volume,Blood,Blood Gas
2,50813,Lactate,Blood,Blood Gas
3,52029,% Ionized Calcium,Blood,Blood Gas
4,50801,Alveolar-arterial Gradient,Blood,Blood Gas
...,...,...,...,...
1617,52249,Delete,Cerebrospinal Fluid,Hematology
1618,52251,Delete,Cerebrospinal Fluid,Hematology
1619,52266,Macrophage,Cerebrospinal Fluid,Hematology
1620,52262,Immunophenotyping,Cerebrospinal Fluid,Hematology


In [103]:
core_labs = [
    'Creatinine',
    'Glucose',
    'Lactate',
    'White Blood Cells',
    'Hemoglobin'
]

lab_items = d_labitems[
    d_labitems['label'].isin(core_labs)
][['itemid', 'label']]


Create a mappin

In [104]:
itemid_to_lab = dict(
    zip(lab_items['itemid'], lab_items['label'])
)


In [105]:
labevents_24h

,hadm_id,itemid,charttime,valuenum,intime,obs_endtime
465,23251352.0,52033,2141-12-17 15:00:00,NaN,2141-12-17 10:24:25,2141-12-18 10:24:25
466,23251352.0,50824,2141-12-17 15:00:00,137.00,2141-12-17 10:24:25,2141-12-18 10:24:25
467,23251352.0,50820,2141-12-17 15:00:00,7.45,2141-12-17 10:24:25,2141-12-18 10:24:25
468,23251352.0,50809,2141-12-17 15:00:00,111.00,2141-12-17 10:24:25,2141-12-18 10:24:25
469,23251352.0,50808,2141-12-17 15:00:00,1.18,2141-12-17 10:24:25,2141-12-18 10:24:25
...,...,...,...,...,...,...
42309,28477357.0,51506,2136-04-22 20:59:00,NaN,2136-04-22 18:01:13,2136-04-23 18:01:13
42310,28477357.0,51484,2136-04-22 20:59:00,NaN,2136-04-22 18:01:13,2136-04-23 18:01:13
42311,22987108.0,50825,2146-06-23 03:59:00,35.90,2146-06-22 11:46:29,2146-06-23 11:46:29
42312,22987108.0,52033,2146-06-23 03:59:00,NaN,2146-06-22 11:46:29,2146-06-23 11:46:29


In [107]:
labs = labevents_24h[
    (labevents_24h['hadm_id'].isin(cohort['hadm_id'])) &
    (labevents_24h['itemid'].isin(itemid_to_lab.keys()))
].copy()


Convert time:

In [108]:
labs['charttime'] = pd.to_datetime(labs['charttime'])


Merge with ICU admission time:

In [110]:
labs

,hadm_id,itemid,charttime,valuenum,intime_x,obs_endtime,intime_y
0,23251352.0,50809,2141-12-17 15:00:00,111.0,2141-12-17 10:24:25,2141-12-18 10:24:25,2141-12-17 10:24:25
1,27089790.0,50912,2140-03-25 18:00:00,1.0,2140-03-25 14:37:26,2140-03-26 14:37:26,2140-03-25 14:37:26
2,27089790.0,51301,2140-03-25 18:00:00,9.0,2140-03-25 14:37:26,2140-03-26 14:37:26,2140-03-25 14:37:26
3,27089790.0,51222,2140-03-25 18:00:00,9.5,2140-03-25 14:37:26,2140-03-26 14:37:26,2140-03-25 14:37:26
4,20755971.0,50912,2115-10-09 21:00:00,3.6,2115-10-09 10:15:25,2115-10-10 10:15:25,2115-10-09 10:15:25
...,...,...,...,...,...,...,...
1315,22675517.0,50811,2114-06-20 09:59:00,10.4,2114-06-20 09:32:58,2114-06-21 09:32:58,2114-06-20 09:32:58
1316,22675517.0,50809,2114-06-20 09:59:00,137.0,2114-06-20 09:32:58,2114-06-21 09:32:58,2114-06-20 09:32:58
1317,22675517.0,51301,2114-06-20 09:59:00,8.0,2114-06-20 09:32:58,2114-06-21 09:32:58,2114-06-20 09:32:58
1318,22675517.0,51222,2114-06-20 09:59:00,10.3,2114-06-20 09:32:58,2114-06-21 09:32:58,2114-06-20 09:32:58


In [115]:
labs = labs.merge(
    first_icustay[['hadm_id', 'intime']],
    on='hadm_id',
    how='left'
)

labs['intime'] = pd.to_datetime(labs['intime'])


In [116]:
labs

,hadm_id,itemid,charttime,valuenum,intime_x,obs_endtime,intime_y,intime
0,23251352.0,50809,2141-12-17 15:00:00,111.0,2141-12-17 10:24:25,2141-12-18 10:24:25,2141-12-17 10:24:25,2141-12-17 10:24:25
1,27089790.0,50912,2140-03-25 18:00:00,1.0,2140-03-25 14:37:26,2140-03-26 14:37:26,2140-03-25 14:37:26,2140-03-25 14:37:26
2,27089790.0,51301,2140-03-25 18:00:00,9.0,2140-03-25 14:37:26,2140-03-26 14:37:26,2140-03-25 14:37:26,2140-03-25 14:37:26
3,27089790.0,51222,2140-03-25 18:00:00,9.5,2140-03-25 14:37:26,2140-03-26 14:37:26,2140-03-25 14:37:26,2140-03-25 14:37:26
4,20755971.0,50912,2115-10-09 21:00:00,3.6,2115-10-09 10:15:25,2115-10-10 10:15:25,2115-10-09 10:15:25,2115-10-09 10:15:25
...,...,...,...,...,...,...,...,...
1315,22675517.0,50811,2114-06-20 09:59:00,10.4,2114-06-20 09:32:58,2114-06-21 09:32:58,2114-06-20 09:32:58,2114-06-20 09:32:58
1316,22675517.0,50809,2114-06-20 09:59:00,137.0,2114-06-20 09:32:58,2114-06-21 09:32:58,2114-06-20 09:32:58,2114-06-20 09:32:58
1317,22675517.0,51301,2114-06-20 09:59:00,8.0,2114-06-20 09:32:58,2114-06-21 09:32:58,2114-06-20 09:32:58,2114-06-20 09:32:58
1318,22675517.0,51222,2114-06-20 09:59:00,10.3,2114-06-20 09:32:58,2114-06-21 09:32:58,2114-06-20 09:32:58,2114-06-20 09:32:58


In [ ]:
labs = labs[
    (labs['charttime'] >= labs['intime']) &
    (labs['charttime'] <= labs['intime'] + pd.Timedelta(hours=24))
]


In [117]:
labs = labs[
    (labs['charttime'] >= labs['intime']) &
    (labs['charttime'] <= labs['intime'] + pd.Timedelta(hours=24))
]


In [118]:
labs['valuenum'] = pd.to_numeric(labs['valuenum'], errors='coerce')
labs = labs.dropna(subset=['valuenum'])

labs['lab_name'] = labs['itemid'].map(itemid_to_lab)


In [119]:
labs_agg = (
    labs
    .groupby(['hadm_id', 'lab_name'])
    .agg(
        mean_value=('valuenum', 'mean'),
        max_value=('valuenum', 'max')
    )
    .reset_index()
)


Pivot to wide format

In [120]:
labs_wide = labs_agg.pivot(
    index='hadm_id',
    columns='lab_name'
)

labs_wide.columns = [
    f"{lab}_{stat}"
    for stat, lab in labs_wide.columns
]

labs_wide = labs_wide.reset_index()


In [121]:
labs_wide

,hadm_id,Creatinine_mean_value,Glucose_mean_value,Hemoglobin_mean_value,Lactate_mean_value,White Blood Cells_mean_value,Creatinine_max_value,Glucose_max_value,Hemoglobin_max_value,Lactate_max_value,White Blood Cells_max_value
0,20044587.0,0.900000,111.333333,9.440000,1.900000,7.666667,1.0,171.0,10.8,2.7,9.7
1,20199380.0,0.600000,116.000000,10.550000,0.900000,11.050000,0.6,116.0,11.0,0.9,11.7
2,20214994.0,0.820000,135.200000,11.087500,8.755556,4.316667,1.0,148.0,13.0,11.0,5.6
3,20291550.0,0.700000,226.000000,10.400000,NaN,7.700000,0.7,226.0,10.4,NaN,7.7
4,20297618.0,1.000000,121.900000,12.460000,2.700000,14.900000,1.1,145.0,13.2,2.7,15.7
...,...,...,...,...,...,...,...,...,...,...,...
95,29366372.0,1.000000,120.625000,11.216667,2.775000,13.533333,1.0,137.0,14.1,3.6,16.2
96,29374560.0,0.900000,132.333333,15.200000,NaN,9.400000,0.9,165.0,15.2,NaN,9.4
97,29642388.0,1.300000,78.000000,11.000000,NaN,10.500000,1.3,78.0,11.0,NaN,10.5
98,29842315.0,7.200000,136.000000,11.300000,2.150000,13.000000,7.2,136.0,11.3,2.3,13.0


In [123]:
labs_wide.to_csv(
    r"C:\Users\user\Desktop\icu_mortality_project\data\interim/labs_agg.csv",
    index=False
)


Merging DataSet

In [124]:
admissions_reduced

,hadm_id,mortality
0,24181354,0
1,25926192,0
2,23983182,0
3,22942076,1
4,21606243,0
...,...,...
270,24745425,0
271,22168393,0
272,27708593,0
273,23251352,0


In [126]:
admissions_reduced['hadm_id'].is_unique


True

In [136]:
cohort = pd.read_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\interim\cohort_with_mortality.csv")
labs_agg = pd.read_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\interim\labs_agg.csv")
vitals_agg = pd.read_csv(r"C:\Users\user\Desktop\icu_mortality_project\data\interim\vitals_agg.csv")
admissions_reduced

,hadm_id,mortality
0,24181354,0
1,25926192,0
2,23983182,0
3,22942076,1
4,21606243,0
...,...,...
270,24745425,0
271,22168393,0
272,27708593,0
273,23251352,0


In [137]:
vitals_agg

,stay_id,dbp_mean_value,heart_rate_mean_value,resp_rate_mean_value,sbp_mean_value,spo2_mean_value,temperature_mean_value,dbp_min_value,heart_rate_min_value,resp_rate_min_value,...,resp_rate_max_value,sbp_max_value,spo2_max_value,temperature_max_value,dbp_last_value,heart_rate_last_value,resp_rate_last_value,sbp_last_value,spo2_last_value,temperature_last_value
0,30057454,54.666667,108.533333,18.266667,79.333333,92.366667,98.057143,47.0,100.0,13.0,...,25.0,91.0,96.0,98.7,55.0,105.0,13.0,79.0,92.0,98.1
1,30101877,69.750000,94.875000,20.541667,137.916667,100.000000,100.071429,49.0,79.0,16.0,...,25.0,166.0,100.0,100.5,76.0,79.0,19.0,147.0,100.0,99.3
2,30458995,65.333333,81.720000,17.600000,129.833333,97.360000,97.966667,45.0,60.0,14.0,...,24.0,157.0,99.0,98.3,60.0,91.0,16.0,143.0,97.0,98.3
3,30585761,59.000000,70.958333,19.375000,139.000000,95.181818,97.950000,59.0,60.0,10.0,...,24.0,139.0,98.0,98.2,59.0,70.0,21.0,139.0,94.0,97.9
4,30665396,NaN,91.346154,20.280000,NaN,98.650000,98.283333,NaN,70.0,10.0,...,36.0,NaN,100.0,98.9,NaN,82.0,22.0,NaN,100.0,98.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,39553978,54.100000,96.500000,20.700000,88.900000,96.300000,98.966667,41.0,91.0,16.0,...,24.0,95.0,99.0,99.5,55.0,94.0,20.0,85.0,95.0,99.5
96,39623478,61.900000,97.000000,15.000000,110.050000,96.550000,98.333333,47.0,84.0,9.0,...,20.0,140.0,99.0,98.7,63.0,95.0,15.0,130.0,93.0,98.7
97,39635619,78.272727,95.352941,21.705882,147.878788,95.735294,100.071429,50.0,82.0,13.0,...,31.0,181.0,100.0,101.3,63.0,82.0,13.0,112.0,94.0,100.8
98,39711498,67.727273,110.960000,20.760000,107.409091,97.000000,99.650000,55.0,29.0,17.0,...,22.0,131.0,100.0,101.7,79.0,121.0,20.0,101.0,100.0,101.7


In [138]:
labs_agg

,hadm_id,Creatinine_mean_value,Glucose_mean_value,Hemoglobin_mean_value,Lactate_mean_value,White Blood Cells_mean_value,Creatinine_max_value,Glucose_max_value,Hemoglobin_max_value,Lactate_max_value,White Blood Cells_max_value
0,20044587.0,0.900000,111.333333,9.440000,1.900000,7.666667,1.0,171.0,10.8,2.7,9.7
1,20199380.0,0.600000,116.000000,10.550000,0.900000,11.050000,0.6,116.0,11.0,0.9,11.7
2,20214994.0,0.820000,135.200000,11.087500,8.755556,4.316667,1.0,148.0,13.0,11.0,5.6
3,20291550.0,0.700000,226.000000,10.400000,NaN,7.700000,0.7,226.0,10.4,NaN,7.7
4,20297618.0,1.000000,121.900000,12.460000,2.700000,14.900000,1.1,145.0,13.2,2.7,15.7
...,...,...,...,...,...,...,...,...,...,...,...
95,29366372.0,1.000000,120.625000,11.216667,2.775000,13.533333,1.0,137.0,14.1,3.6,16.2
96,29374560.0,0.900000,132.333333,15.200000,NaN,9.400000,0.9,165.0,15.2,NaN,9.4
97,29642388.0,1.300000,78.000000,11.000000,NaN,10.500000,1.3,78.0,11.0,NaN,10.5
98,29842315.0,7.200000,136.000000,11.300000,2.150000,13.000000,7.2,136.0,11.3,2.3,13.0


In [139]:
admissions_reduced

,hadm_id,mortality
0,24181354,0
1,25926192,0
2,23983182,0
3,22942076,1
4,21606243,0
...,...,...
270,24745425,0
271,22168393,0
272,27708593,0
273,23251352,0


In [140]:
# Base
df = cohort.copy()

# Add ICU vitals (stay-level)
df = df.merge(
    vitals_agg,
    on='stay_id',
    how='left'
)

# Add admission labs (admission-level)
df = df.merge(
    labs_agg,
    on='hadm_id',
    how='left'
)

# Add mortality label
df = df.merge(
    admissions_reduced,
    on='hadm_id',
    how='left'
)


In [141]:
print(df.shape)
print(cohort.shape)


(100, 46)
(100, 11)


In [ ]:
df['stay_id'].is_unique


True

In [144]:
df.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,...,Glucose_mean_value,Hemoglobin_mean_value,Lactate_mean_value,White Blood Cells_mean_value,Creatinine_max_value,Glucose_max_value,Hemoglobin_max_value,Lactate_max_value,White Blood Cells_max_value,mortality_y
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,...,111.0,11.900000,NaN,4.100000,0.5,115.0,11.9,NaN,4.1,0
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,...,113.0,11.200000,NaN,19.000000,0.4,113.0,11.2,NaN,19.0,0
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,F,46,...,149.0,13.250000,NaN,18.550000,0.8,152.0,13.9,NaN,20.1,0
3,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2156-04-12 16:24:18,2156-04-17 15:57:08,4.981134,F,80,...,123.5,9.750000,1.825000,25.150000,0.9,148.0,10.2,2.4,27.9,0
4,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,290.0,13.766667,3.666667,31.533333,1.6,331.0,14.3,4.4,36.8,0


In [145]:
df = df.rename(columns={'mortality_y': 'mortality'})


In [146]:
df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,...,Glucose_mean_value,Hemoglobin_mean_value,Lactate_mean_value,White Blood Cells_mean_value,Creatinine_max_value,Glucose_max_value,Hemoglobin_max_value,Lactate_max_value,White Blood Cells_max_value,mortality
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,...,111.000000,11.900000,NaN,4.100000,0.5,115.0,11.9,NaN,4.1,0
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,...,113.000000,11.200000,NaN,19.000000,0.4,113.0,11.2,NaN,19.0,0
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,F,46,...,149.000000,13.250000,NaN,18.550000,0.8,152.0,13.9,NaN,20.1,0
3,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2156-04-12 16:24:18,2156-04-17 15:57:08,4.981134,F,80,...,123.500000,9.750000,1.825000,25.150000,0.9,148.0,10.2,2.4,27.9,0
4,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,290.000000,13.766667,3.666667,31.533333,1.6,331.0,14.3,4.4,36.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,10038999,27189241,39711498,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2131-05-22 21:50:33,2131-05-31 17:55:04,8.836470,M,45,...,89.200000,9.533333,1.233333,10.400000,0.9,121.0,9.8,1.6,11.5,0
96,10039708,28258130,33281088,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2140-01-23 18:08:00,2140-02-08 22:28:20,16.180787,F,46,...,232.400000,10.300000,4.500000,11.500000,2.2,315.0,10.7,7.2,13.7,0
97,10039831,26924951,39142259,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2115-12-28 19:36:43,2115-12-30 16:31:48,1.871586,F,57,...,133.666667,11.850000,NaN,13.800000,0.7,176.0,12.1,NaN,15.3,0
98,10039997,24294180,36893762,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2134-09-07 18:03:58,2134-09-08 22:11:05,1.171609,F,67,...,137.000000,10.900000,NaN,8.500000,0.9,137.0,10.9,NaN,8.5,0


In [148]:
df["mortality"].value_counts(normalize=True)

mortality
0    0.89
1    0.11
Name: proportion, dtype: float64

In [154]:
df.drop(["mortality_x"], axis=1)
df.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'gender', 'anchor_age', 'mortality_x',
       'dbp_mean_value', 'heart_rate_mean_value', 'resp_rate_mean_value',
       'sbp_mean_value', 'spo2_mean_value', 'temperature_mean_value',
       'dbp_min_value', 'heart_rate_min_value', 'resp_rate_min_value',
       'sbp_min_value', 'spo2_min_value', 'temperature_min_value',
       'dbp_max_value', 'heart_rate_max_value', 'resp_rate_max_value',
       'sbp_max_value', 'spo2_max_value', 'temperature_max_value',
       'dbp_last_value', 'heart_rate_last_value', 'resp_rate_last_value',
       'sbp_last_value', 'spo2_last_value', 'temperature_last_value',
       'Creatinine_mean_value', 'Glucose_mean_value', 'Hemoglobin_mean_value',
       'Lactate_mean_value', 'White Blood Cells_mean_value',
       'Creatinine_max_value', 'Glucose_max_value', 'Hemoglobin_max_value',
       'Lactate_max_value', 'White Blood Cells_max_value', 'mortalit

In [156]:
df.to_csv(
    r"C:\Users\user\Desktop\icu_mortality_project\data\processed/modeling_table.csv",
    index=False
)


In [157]:
df[['mortality_x', 'mortality']].head()


,mortality_x,mortality
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [158]:
df = df.drop(columns=['mortality_x'])


In [159]:
df['mortality'].value_counts()


mortality
0    89
1    11
Name: count, dtype: int64

In [160]:
df

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,...,Glucose_mean_value,Hemoglobin_mean_value,Lactate_mean_value,White Blood Cells_mean_value,Creatinine_max_value,Glucose_max_value,Hemoglobin_max_value,Lactate_max_value,White Blood Cells_max_value,mortality
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,...,111.000000,11.900000,NaN,4.100000,0.5,115.0,11.9,NaN,4.1,0
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,...,113.000000,11.200000,NaN,19.000000,0.4,113.0,11.2,NaN,19.0,0
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,F,46,...,149.000000,13.250000,NaN,18.550000,0.8,152.0,13.9,NaN,20.1,0
3,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2156-04-12 16:24:18,2156-04-17 15:57:08,4.981134,F,80,...,123.500000,9.750000,1.825000,25.150000,0.9,148.0,10.2,2.4,27.9,0
4,10002495,24982426,36753294,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,M,81,...,290.000000,13.766667,3.666667,31.533333,1.6,331.0,14.3,4.4,36.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,10038999,27189241,39711498,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2131-05-22 21:50:33,2131-05-31 17:55:04,8.836470,M,45,...,89.200000,9.533333,1.233333,10.400000,0.9,121.0,9.8,1.6,11.5,0
96,10039708,28258130,33281088,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2140-01-23 18:08:00,2140-02-08 22:28:20,16.180787,F,46,...,232.400000,10.300000,4.500000,11.500000,2.2,315.0,10.7,7.2,13.7,0
97,10039831,26924951,39142259,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2115-12-28 19:36:43,2115-12-30 16:31:48,1.871586,F,57,...,133.666667,11.850000,NaN,13.800000,0.7,176.0,12.1,NaN,15.3,0
98,10039997,24294180,36893762,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2134-09-07 18:03:58,2134-09-08 22:11:05,1.171609,F,67,...,137.000000,10.900000,NaN,8.500000,0.9,137.0,10.9,NaN,8.5,0


In [161]:
df.to_csv(
    r"C:\Users\user\Desktop\icu_mortality_project\data\processed/modeling_table.csv",
    index=False
)


In [162]:
df.isna().sum()

subject_id                       0
hadm_id                          0
stay_id                          0
first_careunit                   0
last_careunit                    0
intime                           0
outtime                          0
los                              0
gender                           0
anchor_age                       0
dbp_mean_value                   9
heart_rate_mean_value            0
resp_rate_mean_value             0
sbp_mean_value                   9
spo2_mean_value                  0
temperature_mean_value          11
dbp_min_value                    9
heart_rate_min_value             0
resp_rate_min_value              0
sbp_min_value                    9
spo2_min_value                   0
temperature_min_value           11
dbp_max_value                    9
heart_rate_max_value             0
resp_rate_max_value              0
sbp_max_value                    9
spo2_max_value                   0
temperature_max_value           11
dbp_last_value      